<a href="https://colab.research.google.com/github/floranuta/Data_Circle/blob/main/notebooks/model_development.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sprint 2: Model development

In [105]:
# import necessary libraries here
from google.colab import drive
import pandas as pd
from sklearn.preprocessing import OneHotEncoder

In [106]:
# load data
drive.mount("/content/drive")
train_value = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/training_set_values.csv")
train_label = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/training_set_labels.csv")

# merge values and labels on one table
train_data = pd.merge(train_value, train_label, on="id")
train_data

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,id,amount_tsh,date_recorded,funder,gps_height,installer,longitude,latitude,wpt_name,num_private,...,water_quality,quality_group,quantity,quantity_group,source,source_type,source_class,waterpoint_type,waterpoint_type_group,status_group
0,69572,6000.0,14/03/2011,Roman,1390,Roman,34.938093,-9.856322,none,0,...,soft,good,enough,enough,spring,spring,groundwater,communal standpipe,communal standpipe,functional
1,8776,0.0,06/03/2013,Grumeti,1399,GRUMETI,34.698766,-2.147466,Zahanati,0,...,soft,good,insufficient,insufficient,rainwater harvesting,rainwater harvesting,surface,communal standpipe,communal standpipe,functional
2,34310,25.0,25/02/2013,Lottery Club,686,World vision,37.460664,-3.821329,Kwa Mahundi,0,...,soft,good,enough,enough,dam,dam,surface,communal standpipe multiple,communal standpipe,functional
3,67743,0.0,28/01/2013,Unicef,263,UNICEF,38.486161,-11.155298,Zahanati Ya Nanyumbu,0,...,soft,good,dry,dry,machine dbh,borehole,groundwater,communal standpipe multiple,communal standpipe,non functional
4,19728,0.0,13/07/2011,Action In A,0,Artisan,31.130847,-1.825359,Shuleni,0,...,soft,good,seasonal,seasonal,rainwater harvesting,rainwater harvesting,surface,communal standpipe,communal standpipe,functional
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59395,60739,10.0,03/05/2013,Germany Republi,1210,CES,37.169807,-3.253847,Area Three Namba 27,0,...,soft,good,enough,enough,spring,spring,groundwater,communal standpipe,communal standpipe,functional
59396,27263,4700.0,07/05/2011,Cefa-njombe,1212,Cefa,35.249991,-9.070629,Kwa Yahona Kuvala,0,...,soft,good,enough,enough,river,river/lake,surface,communal standpipe,communal standpipe,functional
59397,37057,0.0,11/04/2011,NaN,0,NaN,34.017087,-8.750434,Mashine,0,...,fluoride,fluoride,enough,enough,machine dbh,borehole,groundwater,hand pump,hand pump,functional
59398,31282,0.0,08/03/2011,Malec,0,Musa,35.861315,-6.378573,Mshoro,0,...,soft,good,insufficient,insufficient,shallow well,shallow well,groundwater,hand pump,hand pump,functional


**Task 1: Data cleaning and feature engineering**
- clean data so that data doesn't include null values
- encode categorical values into numerical values
- select features you want to use
- source: https://www.datacamp.com/tutorial/feature-engineering

In [107]:
# gloabal variables

TARGET_COLUMN = "status_group"
TARGET_MAPPING = {"functional": 0, "non functional": 1, "functional needs repair": 2}
FEATURE = ["quantity_group", "region", "payment_type", "extraction_type_class", "management", "quality_group", "pump_age", "status_group"]

In [108]:
class DataCleaner:

    def __init__(self) -> pd.DataFrame:
        """
        This class cleans raw data.

        ----------
        Parameters:
        features: list
          list of features
        """

        self.feature = FEATURE


    def extract_recorded_year(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        """
        Extract year from column recorded_year.

        ----------
        Parameters:
        dataframe: dataFrame

        ---------
        Returns:
        dataframe: dataframe
          dataframe which year_recorded was added after extracting year
        """

        dataframe["date_recorded"] = pd.to_datetime(dataframe['date_recorded'])

        # extract year from date_recorded
        dataframe["year_recorded"] = pd.DatetimeIndex(dataframe["date_recorded"]).year

        return dataframe


    def fill_extracttion_year(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        """
        Fill zero value with mean in a column extraction_year.


        ----------
        Parameters:
        dataframe: dataFrame

        ---------
        Returns:
        dataframe: dataframe
          dataframe after zero was filled with mean of construction_year
        """

        temp_df = dataframe[(dataframe["construction_year"] > 0)]
        mean_consturuction_year = temp_df["construction_year"].mean()

        dataframe["construction_year"] = dataframe['construction_year'].replace(0, mean_consturuction_year)

        return dataframe


    def get_pump_age(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        """
        Calculate pump_age by recorded_year - consturction_year.

        ----------
        Parameters:
        dataframe: dataFrame

        ---------
        Returns:
        dataframe: dataframe
          dataframe that pump_age was added
        """

        # calculate the pump age
        dataframe["pump_age"] = dataframe["year_recorded"] - dataframe["construction_year"]

        return dataframe


    def remove_invalid_records(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        """
        Remove the rows where the pump_age is minus.

        ----------
        Parameters:
        dataframe: dataFrame

        ---------
        Returns:
        dataframe: dataframe
          dataframe after eliminating invailed data
        """

        initial_num_rows = len(dataframe)
        dataframe = dataframe.loc[dataframe["pump_age"] >= 0]
        dataframe = dataframe.reset_index()
        num_removed_row = len(dataframe) - initial_num_rows

        print(f"Removed {num_removed_row} invalid rows (negative pump_age). "
                f"Remaining rows: {len(dataframe)}")

        return dataframe


    def extract_feature(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        """
        Extract necessary features.

        ----------
        Parameters:
        dataframe: dataFrame

        ---------
        Returns:
        dataframe: dataframe
          dataframe contains only necessary features
        """

        return dataframe[self.feature]



    def preprocess(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        """
        Perform preprocessing.

        ----------
        Parameters:
        dataframe: dataFrame
          raw data

        ---------
        Returns:
        dataframe: dataframe
          cleaned dataframe
        """

        dataframe = self.extract_recorded_year(dataframe=dataframe)
        dataframe = self.fill_extracttion_year(dataframe=dataframe)
        dataframe = self.get_pump_age(dataframe=dataframe)
        dataframe = self.remove_invalid_records(dataframe=dataframe)
        cleaned_dataframe = self.extract_feature(dataframe=dataframe)

        print("Data cleaning completed.")

        return cleaned_dataframe

In [109]:
class Encoder:

    def __init__(self):
        """
        This class encodes categorical values in dataframe.

        ----------
        Parameters:
        feature_encoder: Object
          encoder to encode categorical features

        target_column: str or list
          column(s) name of target value

        taregt_encode_mapping: dict
          encoding schema for target value
        """

        self.feature_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
        self.target_column = TARGET_COLUMN
        self.target_encode_mapping = TARGET_MAPPING


    def encode_target(self, categorical_target: pd.DataFrame) -> pd.DataFrame:
        """
        Apply label encoding to taregt column.

        ----------
        Parameters:
        categorical_target: dataFrame
          dataframe of target value

        ---------
        Returns:
        categorical_target: dataframe
          dataframe of encoded target value
        """

        categorical_target = categorical_target.map(self.target_encode_mapping)

        return categorical_target


    def encode_feature(self, categorical_feature: pd.DataFrame) -> pd.DataFrame:
        """
        Apply one-hot encoding to categorical features.
        Reference: https://datasensei.medium.com/how-to-transform-nominal-data-for-ml-with-onehotencoder-from-scikit-learn-f6febfefb3c6

        ----------
        Parameters:
        categorical_feature: dataFrame
          dataframe of feature value

        ---------
        Returns:
        one_hot_feature: dataframe
          dataframe of encoded feature value
        """

        # create a OneHotEncoder that ignores (0 encodes) unseen categories
        # and encode the categorical features for the example dataframe
        X_encoded = self.feature_encoder.fit_transform(categorical_feature)

        # # create the names for the one-hot encoded categorical features
        categorical_columns = [f'{col}_{cat}' for i, col in enumerate(categorical_feature.columns) for cat in self.feature_encoder.categories_[i]]

        # put the features into a dataframe and join with the original
        # numerical features
        one_hot_feature = pd.DataFrame(X_encoded, columns=categorical_columns)

        return one_hot_feature


    def encode_categorcial_value(self, dataframe: pd.DataFrame) -> pd.DataFrame:
        """
        Encode categorical values.

        ----------
        Parameters:
        dataframe: dataFrame
          datframe before encoded

        ---------
        Returns:
        encoded_dataframe: dataframe
          encoded dataframe
        """


        # split the dataframe into its numerical and categorical components
        y_cat = dataframe[self.target_column]
        X_num = dataframe.select_dtypes(exclude='object')
        X_cat = dataframe.select_dtypes(include='object').drop(columns=self.target_column)


        # encode feature column
        X_cat_encoded = self.encode_feature(categorical_feature=X_cat)

        # encode target column
        y_cat_encoded = self.encode_target(categorical_target=y_cat)

        # join all dataframes

        encoded_dataframe = X_num.join(X_cat_encoded).join(y_cat_encoded)

        return encoded_dataframe

In [110]:
# clean data

cleaner = DataCleaner()

cleaned_data = cleaner.preprocess(train_data)
cleaned_data

/tmp/ipython-input-1673652405.py:30: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  dataframe["date_recorded"] = pd.to_datetime(dataframe['date_recorded'])


Removed -9 invalid rows (negative pump_age). Remaining rows: 59391
Data cleaning completed.


,quantity_group,region,payment_type,extraction_type_class,management,quality_group,pump_age,status_group
0,enough,Iringa,annually,gravity,vwc,good,12.000000,functional
1,insufficient,Mara,never pay,gravity,wug,good,3.000000,functional
2,enough,Manyara,per bucket,gravity,vwc,good,4.000000,functional
3,dry,Mtwara,never pay,submersible,vwc,good,27.000000,non functional
4,seasonal,Kagera,never pay,gravity,other,good,14.185314,functional
...,...,...,...,...,...,...,...,...
59386,enough,Kilimanjaro,per bucket,gravity,water board,good,14.000000,functional
59387,enough,Iringa,annually,gravity,vwc,good,15.000000,functional
59388,enough,Mbeya,monthly,handpump,vwc,fluoride,14.185314,functional
59389,insufficient,Dodoma,never pay,handpump,vwc,good,14.185314,functional


In [111]:
# encode cleaned data

encoder = Encoder()

encoded_data = encoder.encode_categorcial_value(cleaned_data)
encoded_data

,pump_age,quantity_group_dry,quantity_group_enough,quantity_group_insufficient,quantity_group_seasonal,quantity_group_unknown,region_Arusha,region_Dar es Salaam,region_Dodoma,region_Iringa,...,management_water board,management_wua,management_wug,quality_group_colored,quality_group_fluoride,quality_group_good,quality_group_milky,quality_group_salty,quality_group_unknown,status_group
0,12.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
1,3.000000,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0
2,4.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
3,27.000000,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1
4,14.185314,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59386,14.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
59387,15.000000,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0
59388,14.185314,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0
59389,14.185314,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0


**Task 2: Implment base pipeline**
- Select ML algorithm you want to use
- Implement algorithm you chosen, K-fold cross validation, training and prediction
- source for K-fold corss validation with code: https://neptune.ai/blog/cross-validation-in-machine-learning-how-to-do-it-right
- source for selecting machine learnig model: https://ujangriswanto08.medium.com/top-algorithms-for-multi-class-classification-in-machine-learning-ed8642ca5440

**Task 3: Implement evaluation mtrix and tune hyperparamter**
- Implement classification report (accuracy, precision, F1 score etc.) and heat map and so on
- Tune hyperparameter
- Select appropriate evaluation matrix and hyperparameter optimization
- source for evaluation matrix: https://www.geeksforgeeks.org/machine-learning/compute-classification-report-and-confusion-matrix-in-python/